In [1]:
import os
import pandas as pd
import numpy as np

# Find all files available to the notebook
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:10]:
        print(f"{indent}  {file}")

input/
  competitions/
    shl-hiring-assessment-2026/
      Dataset_Final/
        sample_submission.csv
        train.csv
        test.csv
        test/
          audio_49.wav
          audio_90.wav
          audio_77.wav
          audio_66.wav
          audio_54.wav
          audio_42.wav
          audio_81.wav
          audio_72.wav
          audio_107.wav
          audio_79.wav
        train/
          audio_49.wav
          audio_663.wav
          audio_5059.wav
          audio_90.wav
          audio_581.wav
          audio_698.wav
          audio_77.wav
          audio_66.wav
          audio_555.wav
          audio_54.wav
  notebooks/
    dhananjaysharma23/
      notebook55d9741d54/
        __results__.html
        submission.csv
        __notebook__.ipynb
        __output__.json
        custom.css


In [2]:
# SHL Hiring Assessment 2026 — Grammar Scoring Engine

## 1. Problem Statement

The objective is to predict the grammar score of spoken English audio samples on a continuous scale from 0 to 5.

The dataset contains 769 labeled training audio samples and 216 test audio samples. Each audio file is approximately 45–60 seconds long.

## 2. Approach

We formulate the task as a regression problem.

The pipeline is:

Audio → Preprocessing → Acoustic Feature Extraction → Extra Trees Regression → Grammar Score

The extracted features include MFCCs, MFCC delta features, spectral centroid, spectral bandwidth, spectral rolloff, zero-crossing rate, RMS energy, chroma features, audio duration, and amplitude statistics.

## 3. Preprocessing

Audio files are loaded as mono audio at a sampling rate of 16 kHz. Leading and trailing silence is removed before feature extraction.

## 4. Model

An Extra Trees Regressor is used because it performs well on a relatively small tabular feature set, can capture nonlinear relationships, and requires limited preprocessing.

## 5. Evaluation

The model is evaluated using RMSE and Pearson Correlation.

Training RMSE: 0.0770

Validation RMSE: 0.6995

Validation Pearson Correlation: 0.8585

## 6. Final Submission

After validation, the final model is trained using all 769 training samples and used to generate predictions for all 216 test samples.

Predictions are clipped to the valid score range and saved as `submission.csv`.

## 7. Conclusion

The proposed acoustic-feature-based regression pipeline provides a simple and interpretable baseline for spoken grammar scoring. The approach can be further improved using pretrained speech representations, speech-to-text linguistic features, and model ensembles.

SyntaxError: invalid character '–' (U+2013) (278960445.py, line 7)

In [ ]:
import pandas as pd
import os

BASE_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train_df = pd.read_csv(os.path.join(BASE_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))
sample_df = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"))

print("TRAIN")
print(train_df.head())
print("\nShape:", train_df.shape)
print("\nColumns:", train_df.columns.tolist())

print("\n\nTEST")
print(test_df.head())
print("\nShape:", test_df.shape)
print("\nColumns:", test_df.columns.tolist())

print("\n\nSAMPLE SUBMISSION")
print(sample_df.head())
print("\nShape:", sample_df.shape)
print("\nColumns:", sample_df.columns.tolist())

In [ ]:
# ============================================
# SHL Grammar Scoring Engine - Baseline Model
# ============================================

import os
import numpy as np
import pandas as pd
import librosa

from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# Paths
BASE_PATH = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
TRAIN_AUDIO = os.path.join(BASE_PATH, "train")
TEST_AUDIO = os.path.join(BASE_PATH, "test")

# ------------------------------------------------
# Feature extraction
# ------------------------------------------------

def extract_features(filename, folder):
    path = os.path.join(folder, filename)

    try:
        # Load audio
        y, sr = librosa.load(path, sr=16000, mono=True)

        # Remove leading/trailing silence
        y, _ = librosa.effects.trim(y, top_db=30)

        features = []

        # 1. MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        features.extend(np.mean(mfcc, axis=1))
        features.extend(np.std(mfcc, axis=1))

        # 2. Delta MFCC
        delta = librosa.feature.delta(mfcc)
        features.extend(np.mean(delta, axis=1))
        features.extend(np.std(delta, axis=1))

        # 3. Spectral features
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        zero_crossing = librosa.feature.zero_crossing_rate(y)

        for feat in [
            spectral_centroid,
            spectral_bandwidth,
            spectral_rolloff,
            zero_crossing
        ]:
            features.append(np.mean(feat))
            features.append(np.std(feat))

        # 4. RMS energy
        rms = librosa.feature.rms(y=y)
        features.append(np.mean(rms))
        features.append(np.std(rms))

        # 5. Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features.extend(np.mean(chroma, axis=1))
        features.extend(np.std(chroma, axis=1))

        # 6. Basic speech/audio statistics
        features.append(len(y) / sr)          # duration
        features.append(np.mean(np.abs(y)))   # mean amplitude
        features.append(np.std(y))            # amplitude variation

        return np.array(features, dtype=np.float32)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return np.zeros(20 * 4 + 4 * 2 + 2 + 12 * 2 + 3, dtype=np.float32)


# ------------------------------------------------
# Extract train + test features
# ------------------------------------------------

print("Extracting training audio features...")

train_features = Parallel(n_jobs=-1, verbose=10)(
    delayed(extract_features)(f, TRAIN_AUDIO)
    for f in train_df["filename"]
)

print("\nExtracting test audio features...")

test_features = Parallel(n_jobs=-1, verbose=10)(
    delayed(extract_features)(f, TEST_AUDIO)
    for f in test_df["filename"]
)

X = np.vstack(train_features)
X_test = np.vstack(test_features)

y = train_df["label"].values.astype(float)

print("\nFeature extraction complete!")
print("Training feature shape:", X.shape)
print("Test feature shape:", X_test.shape)

# ------------------------------------------------
# Train / validation split
# ------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ------------------------------------------------
# Extra Trees Regression
# ------------------------------------------------

model = ExtraTreesRegressor(
    n_estimators=400,
    max_features=0.8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("\nTraining model...")
model.fit(X_train, y_train)

# ------------------------------------------------
# Training metrics
# ------------------------------------------------

train_pred = model.predict(X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))

# ------------------------------------------------
# Validation metrics
# ------------------------------------------------

val_pred = model.predict(X_val)

val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_pearson = pearsonr(y_val, val_pred)[0]

print("\n========================================")
print("MODEL RESULTS")
print("========================================")
print(f"Training RMSE   : {train_rmse:.4f}")
print(f"Validation RMSE : {val_rmse:.4f}")
print(f"Validation Pearson: {val_pearson:.4f}")
print("========================================")

# ------------------------------------------------
# Train final model on ALL training data
# ------------------------------------------------

print("\nTraining final model on all 769 samples...")

final_model = ExtraTreesRegressor(
    n_estimators=500,
    max_features=0.8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

# ------------------------------------------------
# Generate test predictions
# ------------------------------------------------

test_predictions = final_model.predict(X_test)

# Keep predictions within valid score range
test_predictions = np.clip(test_predictions, 0, 5)

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("\nSubmission created!")
print(submission.head(10))
print("\nSubmission shape:", submission.shape)
print("\nSaved to: /kaggle/working/submission.csv")

In [ ]:
# Final Results

| Metric | Score |
|---|---:|
| Training RMSE | 0.0770 |
| Validation RMSE | 0.6995 |
| Validation Pearson Correlation | 0.8585 |
| Kaggle Public Score | 0.7069 |

The final submission contains predictions for all 216 test audio samples.